# Lecture 13 — Hybrid architecture, and closing the loop

**01211271 Industrial AI and IoT** · Electromechanical Manufacturing Engineering

---

Lecture 12 ended with a refitted alarm specification sitting in Colab: twenty-five
numbers, a threshold, a persistence rule and an expiry date. It is useless there.

Today it comes down. That sounds like the easy half of the system and it is not, because
the downlink is the only channel in this entire course that can **make the machine less
safe**. Everything else either adds information or fails quietly. A bad specification
arriving on `@private/model` turns a working protection system into a machine that
nobody is watching — and nothing on the plant floor will tell you.

So the lecture is in two halves, and both of them are about trust.

1. **The downlink.** We will build 19 update messages — two legitimate, the rest
   corrupt, stale, replayed, absurd, or simply wrong — and run them through three apply
   paths. A one-line apply gets **4 of 19** right. Structural
   validation gets **17**. The two it cannot catch are the two
   that are perfectly well formed, and the only thing that catches those is running them.
2. **The sensor.** Every model in this course assumes the accelerometer is working, and
   none of them checks. We will break it four ways and watch a good model answer
   confidently and wrongly — including one case where a faulty sensor makes a **spalled
   bearing look healthier than a healthy machine**.

## Learning objectives

By the end of this notebook you should be able to:

1. Draw the reference architecture end to end and state the division-of-labour rule.
2. Compute the latency budget for a safety-relevant action, and say what round trip
   a cloud-in-the-loop design would have to fit inside.
3. Design a model-update message: version, freshness, integrity, and who approved it.
4. Implement the apply path — validate, stage, verify, commit, acknowledge — and explain
   why the order is not negotiable.
5. Explain why the artefact with *fewer* numbers is the dangerous one to update.
6. Recognise what a faulty sensor does to a good model, and why it is worse than noise.
7. Write a validity gate, and state what it catches, what it costs and what it misses.
8. Enumerate the degraded modes and say what the device does in each — without ever
   stopping deciding.

## Prerequisites

Lecture 10 (the alarm specification), Lecture 11 (the device, and who owns the actuator),
Lecture 12 (the uplink, the retrained spec, and the context bursts — which turn out to
matter here for a reason nobody planned).

In [ ]:
import sys
import copy
import json
import time

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn

print("python     ", sys.version.split()[0])
for m in (np, pd, matplotlib, sklearn):
    print(f"{m.__name__:<11}", m.__version__)

plt.rcParams.update({"figure.figsize": (10.5, 3.6), "font.size": 10,
                     "axes.grid": True, "grid.alpha": 0.35,
                     "axes.spines.top": False, "axes.spines.right": False})
BLUE, ORANGE, AQUA, VIOLET, RED = ("#2a78d6", "#eb6834", "#1baf7a",
                                   "#4a3aa7", "#e34948")
MUTED, YELLOW = "#898781", "#eda100"

## 1. The whole system, and the one rule that holds it together

Six lectures of work, in one sentence:

> **Fast and simple at the edge, slow and rich in the cloud — and the actuator never
> waits for either.**

| | the device | the cloud |
|---|---|---|
| sees | one window, 128 ms | two years, twelve machines |
| decides | in 129 ms, always | in days, sometimes |
| drives | the actuator | a work order |
| fails by | being blind to what it was not trained on | being late |
| survives | the network being down | the device being offline |

The downlink is the arrow that closes the loop, and it is the only one that points
*into* the safety-relevant path. Everything else in this course either adds information
(the uplink) or removes nothing (the dashboard). This one can take protection away.

## 2. The latency budget for a safety action

Before deciding what may be in the loop, work out how much loop there is.

In [ ]:
WINDOW_MS = 128.0            # 256 samples at 2000 Hz
GATE_US = 106.0            # section 8, measured under MicroPython
FEATURES_US = 1143.0         # Lecture 11, measured
CLF_US = 6.0
ALARM_US = 3.0

compute_ms = (GATE_US + FEATURES_US + CLF_US + ALARM_US) / 1000.0
device_ms = WINDOW_MS + compute_ms
print(f"fill the window   {WINDOW_MS:9.2f} ms")
print(f"all the computing {compute_ms:9.2f} ms")
print(f"the device path   {device_ms:9.2f} ms")
print()
print("if the actuator must fire within ...   the round trip you could afford")
for budget in (250, 500, 1000, 2000):
    print(f"   {budget:5d} ms{'':>26}{budget - device_ms:8.1f} ms")

Two things fall out of that table, and the second is the useful one.

First, **the window dominates**. All of the computing — a validity gate, a 256-point
FFT, twelve features, a classifier and an alarm — costs about one percent of the time
spent waiting for the signal to exist. Optimising the code is not how you make this
device faster; shortening the window is, and that costs frequency resolution.

Second, and this is the whole point: the device has already spent
129 ms of whatever budget the application allows. What is left is what a cloud
round trip would have to fit inside — **at the tail, not at the mean**. A design that
works at the median round trip and fails at the 99th percentile is a design that fails
about fifteen times an hour at this window rate.

> Nothing in this course puts the cloud in the loop, and this table is why. The cloud
> proposes; the device disposes.

## 3. The update message, and the apply path

A downlink message is not a data message. A data message that gets lost costs you one
window of history; an update message that is wrong costs you the machine. The schema
reflects that.

| field | why it is there |
|---|---|
| `v` | schema version — the device refuses what it does not speak |
| `kind` | `"alarm"` or `"classifier"`; they are gated very differently (section 6) |
| `ver` | monotonic. A device that accepts an older version can be rolled back by a replay |
| `t`, `ttl` | issued at, and shelf life. Lecture 10 put an expiry date on the spec; this enforces it |
| `sum` | a digest over the payload. Catches corruption — **not** malice; say which one you are relying on |
| `by` | the human who approved it. Lecture 12 showed a retrain can silently make things worse |
| `payload` | 25 numbers, or 39 |

And the apply path, in the order that matters:

```
validate  →  stage  →  verify  →  commit  →  ack
```

* **validate** — structure, version, freshness, integrity, and numbers in sane ranges.
* **stage** — into a *standby* slot. Nothing that is running gets touched.
* **verify** — run the candidate, in shadow, on windows the device kept.
* **commit** — one pointer assignment. A power cut leaves the old spec, not half of two.
* **ack** — say what you did and why, back up the link.

Any of the five may refuse, and refusing is not a failure: the device keeps running the
specification it already had.

In [ ]:
TOPIC_MODEL = "@private/model"        # downlink: NETPIE private topic
TOPIC_ACK = "@msg/ack"                # the device says what it did

SCHEMA_VERSION = 1
N_FEATURES = 12
N_CLASSES = 3

# What a sane alarm specification looks like.  These are not tastes; each one is
# a property the device can check in microseconds and each one has been violated
# by a real update somewhere.
THRESHOLD_RANGE = (2.0, 12.0)         # a max-|z| threshold outside this is wrong
MIN_SD = 1e-6                         # a zero standard deviation is a divide by 0
MAX_AGE_S = 30 * 86400                # a spec issued a month ago is stale
MAX_SKEW_S = 3600                     # issued in the future by more than an hour


# ------------------------------------------------------------ the message
def checksum(payload):
    """A short digest over the canonical payload.

    Not a signature -- it catches corruption, not malice, and the difference
    matters enough that the device logs which one it is relying on.
    """
    blob = json.dumps(payload, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(blob.encode()).hexdigest()[:16]


def make_update(kind, payload, version, issued_at, expires_in=MAX_AGE_S,
                fitted_on=None, approved_by=None):
    """One downlink message.

    kind        "alarm" or "classifier" -- which artefact this replaces
    version     monotonically increasing.  The device refuses to go backwards.
    issued_at   unix seconds, from the cloud's clock
    expires_in  after this the device stops trusting it
    fitted_on   how many windows, and from when -- for the log, not the device
    approved_by a named human.  Lecture 12 showed a retrain can silently make
                things worse; somebody has to own that.
    """
    return {"v": SCHEMA_VERSION, "kind": kind, "ver": int(version),
            "t": float(issued_at), "ttl": float(expires_in),
            "fitted_on": fitted_on, "by": approved_by,
            "sum": checksum(payload), "payload": payload}

In [ ]:
class Reject(Exception):
    """Why the device refused.  The reason goes in the ack, and in the log."""


def _finite(a):
    return all(isinstance(v, (int, float)) and math.isfinite(v) for v in a)


def validate(msg, current_ver, now, n_features=N_FEATURES, n_classes=N_CLASSES):
    """Every check the device makes before an update is allowed anywhere near
    the running alarm.  Raises Reject with a reason; returns the payload.

    Read it as a list of things that have gone wrong in the field:
    """
    if not isinstance(msg, dict):
        raise Reject("not an object")
    for k in ("v", "kind", "ver", "t", "sum", "payload"):
        if k not in msg:
            raise Reject(f"missing field {k}")
    if msg["v"] != SCHEMA_VERSION:
        raise Reject(f"schema v{msg['v']}, device speaks v{SCHEMA_VERSION}")
    if msg["kind"] not in ("alarm", "classifier"):
        raise Reject(f"unknown kind {msg['kind']!r}")
    if checksum(msg["payload"]) != msg["sum"]:
        raise Reject("checksum mismatch -- truncated or corrupted")
    if not isinstance(msg["ver"], int) or msg["ver"] <= current_ver:
        raise Reject(f"version {msg['ver']} not newer than {current_ver}")
    if msg["t"] > now + MAX_SKEW_S:
        raise Reject("issued in the future -- check the cloud's clock")
    if now - msg["t"] > msg.get("ttl", MAX_AGE_S):
        raise Reject("expired -- a spec has a shelf life")

    p = msg["payload"]
    if msg["kind"] == "alarm":
        for k in ("scaler_mean", "scaler_scale", "threshold", "persistence",
                  "feature_names"):
            if k not in p:
                raise Reject(f"alarm spec missing {k}")
        mu, sd = p["scaler_mean"], p["scaler_scale"]
        if len(mu) != n_features or len(sd) != n_features:
            raise Reject(f"expected {n_features} features, got {len(mu)}/{len(sd)}")
        if not (_finite(mu) and _finite(sd)):
            raise Reject("non-finite baseline -- NaN or inf in the numbers")
        if min(sd) < MIN_SD:
            raise Reject("a standard deviation is zero -- divide by zero on device")
        thr = p["threshold"]
        if not isinstance(thr, (int, float)) or not math.isfinite(thr):
            raise Reject("threshold is not a finite number")
        if not (THRESHOLD_RANGE[0] <= thr <= THRESHOLD_RANGE[1]):
            raise Reject(f"threshold {thr:.4g} outside {THRESHOLD_RANGE}")
        m, n = p["persistence"]["m"], p["persistence"]["n"]
        if not (1 <= m <= n <= 64):
            raise Reject(f"persistence {m}-of-{n} is not sane")
        if list(p["feature_names"]) != list(EXPECTED_FEATURES):
            raise Reject("feature order does not match the device's")
    else:
        for k in ("coef", "intercept", "classes"):
            if k not in p:
                raise Reject(f"classifier missing {k}")
        W, b = p["coef"], p["intercept"]
        if len(W) != n_classes or len(b) != n_classes:
            raise Reject(f"expected {n_classes} classes, got {len(W)}/{len(b)}")
        if any(len(r) != n_features for r in W):
            raise Reject("a coefficient row is the wrong length")
        if not all(_finite(r) for r in W) or not _finite(b):
            raise Reject("non-finite coefficients")
        if all(all(v == 0.0 for v in r) for r in W):
            raise Reject("all coefficients zero -- the model predicts nothing")
    return p

In [ ]:
EXPECTED_FEATURES = ["mean", "rms", "std", "ptp", "crest", "kurt", "zcr",
                     "dom_freq", "e_1x", "e_2x", "e_bpfo", "e_hi"]


class SpecStore:
    """Two slots and a pointer.  The oldest idea in embedded updates.

    `active` is what the alarm is using this second.  `standby` is where a
    candidate lands.  Commit is one pointer assignment, so a power cut during an
    update leaves the device running the old spec rather than half of two.

    `last_good` is the spec the device will fall back to if the new one turns
    out to misbehave after commit -- which is a different failure from an
    invalid one, and the reason `verify()` exists.
    """

    def __init__(self, alarm, classifier, ver_alarm=1, ver_clf=1):
        self.alarm = alarm
        self.classifier = classifier
        self.ver = {"alarm": ver_alarm, "classifier": ver_clf}
        self.last_good = {"alarm": copy.deepcopy(alarm),
                          "classifier": copy.deepcopy(classifier)}
        self.standby = {}
        self.log = []

    # -- the three steps, kept separate on purpose -------------------------
    def stage(self, msg, now):
        """Validate into the standby slot.  Nothing running is touched."""
        p = validate(msg, self.ver[msg["kind"]], now)
        self.standby[msg["kind"]] = (msg["ver"], copy.deepcopy(p), msg)
        return p

    def commit(self, kind, on_commit=None):
        """One assignment.  Atomic as far as the alarm is concerned."""
        if kind not in self.standby:
            raise Reject("nothing staged")
        ver, p, msg = self.standby.pop(kind)
        self.last_good[kind] = copy.deepcopy(getattr(self, kind))
        setattr(self, kind, p)
        self.ver[kind] = ver
        if on_commit is not None:
            on_commit(kind)          # e.g. reset the persistence ring buffer
        self.log.append({"event": "commit", "kind": kind, "ver": ver,
                         "by": msg.get("by")})
        return p

    def rollback(self, kind, reason):
        setattr(self, kind, copy.deepcopy(self.last_good[kind]))
        self.ver[kind] -= 1
        self.log.append({"event": "rollback", "kind": kind, "reason": reason})

    def apply(self, msg, now, on_commit=None):
        """stage -> commit, with the reason recorded either way.

        Returns (accepted, reason).  Never raises: a device that crashes on a
        bad message is a device anyone can switch off from the internet.
        """
        try:
            self.stage(msg, now)
        except Reject as e:
            self.log.append({"event": "reject", "kind": msg.get("kind"),
                             "reason": str(e)})
            return False, str(e)
        except Exception as e:                       # malformed beyond parsing
            self.log.append({"event": "reject", "kind": None,
                             "reason": f"malformed: {type(e).__name__}"})
            return False, f"malformed: {type(e).__name__}"
        self.commit(msg["kind"], on_commit=on_commit)
        return True, "committed"

Read `apply()` once more and notice what it does **not** do: it does not raise. A device
that crashes on a malformed message can be switched off by anybody who can publish to
its topic, and on a public broker that is a larger set of people than you think.

Here is the one-line version, for comparison — the thing almost everyone writes first.

In [ ]:
def apply_naive(store, msg):
    """What most first attempts look like.  One line, no checks.

    It is here to be measured, not to be copied.
    """
    try:
        store.__dict__[msg["kind"]] = msg["payload"]
        store.ver[msg["kind"]] = msg["ver"]
        return True, "applied"
    except Exception as e:
        return False, f"crashed: {type(e).__name__}"

## 4. The gauntlet

19 messages arrive on `@private/model`. Two of them are legitimate and **must be
accepted** — a gate that refuses everything is not a gate, it is a disconnected wire.
The rest are things that have actually turned up on a downlink topic somewhere.

In [ ]:
"""
rig.py — the rotor-rig simulator shared by Lectures 8-14
01211271 Industrial AI and IoT

Synthetic but physics-motivated vibration data for a small motor-driven rotor rig.

Rig model
---------
A 3-phase induction motor drives a rotor disc supported by two rolling-element
bearings.  A MEMS accelerometer is mounted radially on the drive-end bearing
housing and sampled at fs = 2000 Hz.

Three machine states are simulated:

  normal    : residual unbalance only -> modest 1x shaft-rate component
  imbalance : added trial mass on the disc -> large 1x component
  bearing   : outer-race spall -> periodic impulses at BPFO that ring the
              bearing housing resonance

Outer-race defect frequency, for a bearing with n = 9 balls, ball/pitch
diameter ratio d/D = 0.2 and contact angle 0:

    BPFO = (n/2) * (1 - (d/D) cos(phi)) * f_r = 4.5 * 0.8 * f_r = 3.6 * f_r

so at a nominal 30 Hz shaft rate (1800 rpm) the impulses arrive at 108 Hz.

Every run carries its own nuisance variation -- shaft speed, sensor gain,
broadband noise floor, DC bias and a structural tone.  This is what makes a
random train/test split across overlapping windows dishonest: the model can
memorise the run, not the fault.

Lecture 8 built the classification dataset from make_dataset().  Lecture 9 adds
make_speed_sweep(), a separate acquisition campaign used for the soft-sensor
thread: the rig is run across its speed range in the healthy state so that a
regression model can learn to read shaft speed off the vibration alone.
"""

import numpy as np

# ----------------------------------------------------------------- constants
FS = 2000.0                 # sampling rate, Hz
RUN_SECONDS = 4.0           # length of one acquisition run
RUNS_PER_CLASS = 16
CLASSES = ("normal", "imbalance", "bearing")

WIN = 256                   # default window length, samples (128 ms)
HOP = 128                   # 50 % overlap

F_RESONANCE = 600.0         # bearing housing resonance, Hz
TAU_RING = 0.0012           # impulse ring-down time constant, s
BPFO_RATIO = 3.6            # outer-race defect order

# Fixed analysis bands, Hz.  Shaft speed wanders over 28-32 Hz, so the bands
# are wide enough to hold the order they are named for without a tachometer.
BANDS = {
    "e_1x":   (20.0, 42.0),     # shaft rate
    "e_2x":   (48.0, 78.0),     # twice shaft rate
    "e_bpfo": (90.0, 132.0),    # outer-race defect order
    "e_hi":   (400.0, 900.0),   # housing resonance region
}

FEATURE_NAMES = [
    "mean", "rms", "std", "ptp", "crest", "kurt", "zcr",
    "dom_freq", "e_1x", "e_2x", "e_bpfo", "e_hi",
]


# ------------------------------------------------------------ signal synthesis
def _run_params(rng, state):
    """Nuisance parameters that vary from one acquisition run to the next."""
    return dict(
        f_r=rng.uniform(28.0, 32.0),          # shaft rate, Hz
        gain=rng.uniform(0.85, 1.25),         # sensor / mounting gain
        noise=rng.uniform(0.04, 0.20),        # broadband floor, g rms
        bias=rng.uniform(-0.06, 0.06),        # accelerometer DC offset, g
        f_struct=rng.uniform(150.0, 700.0),   # frame resonance -- a confounder
        a_struct=rng.uniform(0.03, 0.15),
        a_1x=(rng.uniform(0.20, 0.38) if state != "imbalance"
              else rng.uniform(0.58, 1.10)),
        a_imp=(rng.uniform(0.45, 1.40) if state == "bearing" else 0.0),
    )


def _synth(rng, p, seconds=RUN_SECONDS, fs=FS):
    """Turn a parameter dict into a waveform.  Shared by every campaign."""
    n = int(seconds * fs)
    t = np.arange(n) / fs

    ph = rng.uniform(0, 2 * np.pi, 4)
    x = (p["a_1x"] * np.sin(2 * np.pi * p["f_r"] * t + ph[0])
         + 0.32 * p["a_1x"] * np.sin(2 * np.pi * 2 * p["f_r"] * t + ph[1])
         + 0.12 * p["a_1x"] * np.sin(2 * np.pi * 3 * p["f_r"] * t + ph[2])
         + p["a_struct"] * np.sin(2 * np.pi * p["f_struct"] * t + ph[3]))

    if p["a_imp"] > 0:
        f_bpfo = BPFO_RATIO * p["f_r"]
        period = fs / f_bpfo
        k = 0
        while True:
            # 1 % random slip, as real rolling elements do
            idx = int(k * period * (1.0 + rng.normal(0, 0.01)))
            if idx >= n:
                break
            tail = np.arange(n - idx) / fs
            ring = (p["a_imp"] * rng.uniform(0.75, 1.25)
                    * np.exp(-tail / TAU_RING)
                    * np.sin(2 * np.pi * F_RESONANCE * tail))
            x[idx:] += ring
            k += 1

    x = p["gain"] * (x + rng.normal(0, p["noise"], n)) + p["bias"]
    return x.astype(np.float64)


def make_run(rng, state, seconds=RUN_SECONDS, fs=FS):
    """Synthesise one acquisition run.  Returns (signal, params)."""
    p = _run_params(rng, state)
    return _synth(rng, p, seconds, fs), p


def make_dataset(seed=7, runs_per_class=RUNS_PER_CLASS):
    """All runs.  Returns list of dicts with signal, label and run id."""
    rng = np.random.default_rng(seed)
    runs, rid = [], 0
    for state in CLASSES:
        for _ in range(runs_per_class):
            x, p = make_run(rng, state)
            runs.append({"x": x, "label": state, "run": rid, "params": p})
            rid += 1
    return runs


# ------------------------------------------------------------------- features
def frame(x, win=WIN, hop=HOP):
    """Slice a 1-D signal into overlapping windows -> (n_windows, win)."""
    n = 1 + (len(x) - win) // hop
    idx = np.arange(win)[None, :] + hop * np.arange(n)[:, None]
    return x[idx]


def time_features(w):
    """Seven time-domain features for each row of w."""
    mean = w.mean(axis=1)
    ac = w - mean[:, None]                      # remove DC before anything else
    rms = np.sqrt((ac ** 2).mean(axis=1))
    std = ac.std(axis=1)
    ptp = np.ptp(w, axis=1)
    peak = np.abs(ac).max(axis=1)
    crest = peak / np.maximum(rms, 1e-12)
    m4 = (ac ** 4).mean(axis=1)
    kurt = m4 / np.maximum(rms, 1e-12) ** 4     # non-excess kurtosis; 3.0 = Gaussian
    zc = np.diff(np.signbit(ac).astype(np.int8), axis=1)
    zcr = np.abs(zc).sum(axis=1) / (w.shape[1] - 1)
    return np.column_stack([mean, rms, std, ptp, crest, kurt, zcr])


def freq_features(w, fs=FS, normalise=True):
    """Dominant frequency plus four band energies.

    normalise=True gives each band as a fraction of the window's total power,
    which makes the feature independent of sensor gain.  normalise=False gives
    the absolute band power, whose units and magnitude differ wildly from the
    time-domain features -- useful for showing when feature scaling matters.
    """
    n = w.shape[1]
    win = np.hanning(n)
    ac = w - w.mean(axis=1, keepdims=True)
    spec = np.abs(np.fft.rfft(ac * win, axis=1)) ** 2
    f = np.fft.rfftfreq(n, 1.0 / fs)
    spec[:, 0] = 0.0                            # DC carries no information here
    total = np.maximum(spec.sum(axis=1), 1e-20)

    dom = f[spec.argmax(axis=1)]
    cols = [dom]
    for lo, hi in BANDS.values():
        m = (f >= lo) & (f < hi)
        band = spec[:, m].sum(axis=1)
        cols.append(band / total if normalise else band)
    return np.column_stack(cols)


def features_of(w, fs=FS, normalise=True):
    """Full 12-feature vector for each row of w."""
    return np.column_stack([time_features(w), freq_features(w, fs, normalise)])


def build_table(runs, win=WIN, hop=HOP, fs=FS, normalise=True):
    """Turn the raw runs into the (X, y, groups) learning problem."""
    X, y, g = [], [], []
    for r in runs:
        w = frame(r["x"], win, hop)
        X.append(features_of(w, fs, normalise))
        y += [r["label"]] * w.shape[0]
        g += [r["run"]] * w.shape[0]
    return np.vstack(X), np.array(y), np.array(g)


def raw_table(runs, win=WIN, hop=HOP):
    """The naive alternative: the raw samples of each window as the features."""
    X, y, g = [], [], []
    for r in runs:
        w = frame(r["x"], win, hop)
        X.append(w)
        y += [r["label"]] * w.shape[0]
        g += [r["run"]] * w.shape[0]
    return np.vstack(X), np.array(y), np.array(g)


# ------------------------------------------------- Lecture 9: the speed sweep
SWEEP_RUNS = 40
SWEEP_FR = (22.0, 40.0)     # stays inside the e_1x band (20-42 Hz)


def make_speed_sweep(seed=11, n_runs=SWEEP_RUNS, seconds=RUN_SECONDS):
    """A healthy-machine speed sweep, for the virtual tachometer.

    The rig has no tachometer.  To build one in software you run the machine
    across its speed range in a known-good state and record the true shaft
    speed from the drive's own setpoint.  Each run still carries its own gain,
    noise floor, bias and frame resonance -- the soft sensor has to work in
    spite of those, not because of them.

    Returns a list of dicts with the signal, the true shaft speed and a run id.
    """
    rng = np.random.default_rng(seed)
    runs = []
    for rid in range(n_runs):
        p = _run_params(rng, "normal")
        p["f_r"] = float(rng.uniform(*SWEEP_FR))
        x = _synth(rng, p, seconds, FS)
        runs.append({"x": x, "f_r": p["f_r"],
                     "run": rid, "params": p})
    return runs


def build_speed_table(runs, win=WIN, hop=HOP, fs=FS):
    """(X, y, groups) for the soft sensor.  y is the true shaft speed in Hz."""
    X, y, g = [], [], []
    for r in runs:
        w = frame(r["x"], win, hop)
        X.append(features_of(w, fs))
        y += [r["f_r"]] * w.shape[0]
        g += [r["run"]] * w.shape[0]
    return np.vstack(X), np.array(y), np.array(g)


# =========================================== Lecture 10: the fleet campaign
# A year of monitoring across a fleet of nominally identical rigs.  Faults are
# RARE, and the ones you catch early are mild -- which is the whole point of
# catching them early, and the reason recall is hard.
FLEET_N = {"normal": 132, "imbalance": 12, "bearing": 6}


def _fleet_params(rng, state):
    """Run parameters for the fleet.  Fault severity spans incipient to severe."""
    p = _run_params(rng, state)
    if state == "imbalance":
        # a light trial mass barely shifts 1x; a thrown blade is unmistakable
        p["a_1x"] = float(rng.uniform(0.42, 1.10))
    if state == "bearing":
        # an early spall rings faintly; a spalled race is loud
        p["a_imp"] = float(rng.uniform(0.22, 1.40))
    return p


def make_fleet(seed=23, counts=None, seconds=RUN_SECONDS, fs=FS):
    """One acquisition per machine per month, across a fleet.

    Returns runs in a shuffled order with a 'label' and a 'severity' field.
    Prevalence is industrial, not academic: most machines are fine, a few have
    imbalance, and bearing spalls are rarer still.
    """
    counts = counts or FLEET_N
    rng = np.random.default_rng(seed)
    runs, rid = [], 0
    for state, n in counts.items():
        for _ in range(n):
            p = _fleet_params(rng, state)
            x = _synth(rng, p, seconds, fs)
            sev = (p["a_1x"] if state == "imbalance"
                   else p["a_imp"] if state == "bearing" else 0.0)
            runs.append({"x": x, "label": state, "run": rid,
                         "severity": float(sev), "params": p})
            rid += 1
    order = rng.permutation(len(runs))
    return [runs[i] for i in order]


# ============================================ Lecture 10: the drift campaign
# The same healthy machine, measured monthly for four years.  Nothing breaks --
# but the sensor mount relaxes, the ambient noise floor rises and the frame
# resonance walks.  A threshold set in the first year will not survive.
def make_drift_campaign(seed=31, n_months=48, fault_from=42, seconds=RUN_SECONDS,
                        fs=FS):
    """Healthy runs in time order with a slow baseline drift, then a real fault.

    Each run carries 'month'.  Runs from `fault_from` onward are a genuine
    bearing spall, so the last months contain something a detector SHOULD fire
    on -- everything before that is a healthy machine that merely looks
    different from how it looked in month 0.
    """
    rng = np.random.default_rng(seed)
    runs = []
    for m in range(n_months):
        t = m / (n_months - 1)                       # 0 -> 1 over the campaign
        faulty = m >= fault_from
        p = _run_params(rng, "bearing" if faulty else "normal")
        p["noise"] = float(0.06 + 0.050 * t + rng.normal(0, 0.005))
        p["a_struct"] = float(0.04 + 0.045 * t + rng.normal(0, 0.004))
        p["f_struct"] = float(180.0 + 110.0 * t + rng.normal(0, 8.0))
        p["gain"] = float(1.0 + 0.10 * t + rng.normal(0, 0.015))
        if faulty:
            p["a_imp"] = 1.10
        x = _synth(rng, p, seconds, fs)
        runs.append({"x": x, "label": "bearing" if faulty else "normal",
                     "run": m, "month": m, "params": p})
    return runs


# ======================================== Lecture 12: one shift, continuously
# Lectures 8-11 worked in 4-second acquisitions.  A device that is deployed does
# not take acquisitions -- it runs.  This campaign is one machine monitored
# without interruption through a shift, so that "how often should it publish?"
# becomes a question with a measurable answer.
#
# Two things happen during the shift and only one of them is a fault:
#
#   * a LOAD CHANGE at `load_at_min` -- the operator takes a heavier cut.  The
#     machine is fine.  Vibration amplitude rises anyway, which is exactly what
#     a fixed dashboard threshold on rms will fire on.
#   * a BEARING SPALL from `fault_at_min`, growing in severity.  This is the
#     thing worth telling anybody about.
SHIFT_SEG = 4.0             # seconds of signal synthesised at a time


def make_shift(seed=53, minutes=20.0, load_at_min=5.0, load_until_min=9.0,
               fault_at_min=12.0, fs=FS, win=WIN, hop=HOP):
    """A continuous run of one machine.  Returns the FEATURE STREAM, not samples.

    Returns a dict with:
        X          (n_windows, 12) features, in FEATURE_NAMES order
        t          window start time, seconds from the start of the shift
        state      per-window ground truth: 'normal' | 'loaded' | 'bearing'
        fault_at   seconds at which the spall begins
        load_at    (start, end) seconds of the heavy cut
        hop_s      seconds between consecutive windows
    """
    rng = np.random.default_rng(seed)
    n_seg = int(round(minutes * 60.0 / SHIFT_SEG))
    base = _run_params(rng, "normal")

    X, tt, st = [], [], []
    for k in range(n_seg):
        t0 = k * SHIFT_SEG
        p = dict(base)
        # slow wander that a real machine has and a 4-second acquisition hides
        p["f_r"] = base["f_r"] + 0.6 * np.sin(2 * np.pi * t0 / 420.0) + rng.normal(0, 0.05)
        p["noise"] = base["noise"] * float(rng.uniform(0.94, 1.06))

        loaded = load_at_min * 60.0 <= t0 < load_until_min * 60.0
        if loaded:
            p["a_1x"] = base["a_1x"] * 1.85       # heavier cut, healthy machine
            p["gain"] = base["gain"] * 1.10

        sev = 0.0
        if t0 >= fault_at_min * 60.0:
            # the spall grows: barely audible at onset, unmistakable by the end
            frac = (t0 - fault_at_min * 60.0) / max(
                1e-9, minutes * 60.0 - fault_at_min * 60.0)
            sev = 0.18 + 1.05 * frac
            p["a_imp"] = float(sev)

        x = _synth(rng, p, SHIFT_SEG, fs)
        w = frame(x, win, hop)
        f = features_of(w, fs)
        X.append(f)
        tt.append(t0 + np.arange(len(f)) * hop / fs)
        st += [("bearing" if sev > 0 else "loaded" if loaded else "normal")] * len(f)

    return {"X": np.vstack(X), "t": np.concatenate(tt), "state": np.array(st),
            "fault_at": fault_at_min * 60.0,
            "load_at": (load_at_min * 60.0, load_until_min * 60.0),
            "hop_s": hop / fs, "minutes": minutes}


# ==================================== Lecture 12: a fleet, measured for 2 years
# One acquisition per machine per month.  Two effects are superimposed and
# telling them apart is the whole cloud-side argument:
#
#   * a PLANT-WIDE ambient change -- every machine's noise floor creeps up as
#     the building fills with new equipment.  Common mode.  Nothing is wrong.
#   * ONE machine developing a bearing spall so slowly that no single monthly
#     acquisition looks alarming until it is nearly too late.
MONTH_MACHINES = 12
MONTH_MONTHS = 24


def make_month_campaign(seed=61, n_machines=MONTH_MACHINES, n_months=MONTH_MONTHS,
                        degrader=7, degrade_from=8, seconds=RUN_SECONDS, fs=FS):
    """Monthly acquisitions from a small fleet.  Returns a list of run dicts.

    Each run carries 'machine', 'month', 'label' and 'severity'.
    """
    rng = np.random.default_rng(seed)
    # every machine has its own fixed personality: mounting, gain, resonance
    ident = []
    for m in range(n_machines):
        q = _run_params(rng, "normal")
        ident.append(q)

    runs = []
    for month in range(n_months):
        u = month / (n_months - 1)
        ambient = 1.0 + 0.38 * u                 # common mode, plant-wide
        for m in range(n_machines):
            p = dict(ident[m])
            p["noise"] = float(ident[m]["noise"] * ambient
                               * rng.uniform(0.96, 1.04))
            p["f_r"] = float(ident[m]["f_r"] + rng.normal(0, 0.25))
            p["gain"] = float(ident[m]["gain"] * rng.uniform(0.97, 1.03))
            p["a_1x"] = float(ident[m]["a_1x"] * rng.uniform(0.94, 1.06))

            sev = 0.0
            if m == degrader and month >= degrade_from:
                frac = (month - degrade_from) / max(1, n_months - 1 - degrade_from)
                sev = 0.15 + 1.35 * frac
                p["a_imp"] = float(sev)
            x = _synth(rng, p, seconds, fs)
            runs.append({"x": x, "machine": m, "month": month,
                         "label": "bearing" if sev > 0 else "normal",
                         "severity": float(sev), "params": p})
    return runs


def build_month_table(runs, win=WIN, hop=HOP, fs=FS):
    """One row per window, with machine and month carried alongside."""
    X, mach, mon, lab, sev = [], [], [], [], []
    for r in runs:
        f = features_of(frame(r["x"], win, hop), fs)
        X.append(f)
        mach += [r["machine"]] * len(f)
        mon += [r["month"]] * len(f)
        lab += [r["label"]] * len(f)
        sev += [r["severity"]] * len(f)
    return (np.vstack(X), np.array(mach), np.array(mon), np.array(lab),
            np.array(sev))

In [ ]:
import rig
import downlink as dl
from downlink import (SpecStore, apply_naive, apply_verified, hostile_updates,
                      alarm_rate, outcome_of, make_update)

alarm0 = json.load(open("lecture10_alarm.json"))      # what the device is running
clf0 = json.load(open("lecture9_model.json"))

sh = rig.make_shift(seed=53, minutes=20.0)
t, X, state = sh["t"], sh["X"], sh["state"]
HEALTHY = X[state == "normal"]
LOADED = X[state == "loaded"]
FAULTY = X[t >= 17 * 60]

mu0, sd0 = np.array(alarm0["scaler_mean"]), np.array(alarm0["scaler_scale"])
QUIET = HEALTHY[-1200:]                                # a stretch nothing happened in
ALARMED = FAULTY[np.abs((FAULTY - mu0) / sd0).max(1) > alarm0["threshold"]][:400]
print(f"the device kept {len(QUIET)} quiet windows and {len(ALARMED)} that made it raise")

In [ ]:
def fit_alarm(Xf, q=0.99):
    '''The Lecture 10 recipe, applied to whatever windows you hand it.'''
    m_, s_ = Xf.mean(0), Xf.std(0)
    s_ = np.where(s_ > 0, s_, 1.0)
    z = np.abs((Xf - m_) / s_).max(1)
    return {"scaler_mean": m_.tolist(), "scaler_scale": s_.tolist(),
            "threshold": float(np.quantile(z, q)), "threshold_quantile": q,
            "persistence": {"m": 3, "n": 5},
            "feature_names": list(dl.EXPECTED_FEATURES),
            "fitted_on_healthy_windows": int(len(Xf)), "review_after_months": 12}


good_alarm = fit_alarm(HEALTHY[:4000])                 # the update we actually want
overfit_alarm = fit_alarm(np.vstack([HEALTHY[:4000], FAULTY]))   # absorbed the fault
loaded_alarm = fit_alarm(LOADED)                       # refitted during a heavy cut

NOW, CUR_VER = 1789000000.0, 3
CASES = hostile_updates(good_alarm, clf0, NOW, CUR_VER, old_alarm=alarm0,
                        overfit_alarm=overfit_alarm, loaded_alarm=loaded_alarm)
print(f"{len(CASES)} messages, of which "
      f"{sum(1 for c in CASES if c[2] == 'accept')} must be accepted\n")
for name, msg, expect in CASES:
    print(f"  {expect:>7}   {name}")

The three apply paths, and one measurement that is not "did it parse" but **what is the
machine left with**: does the device still raise on a genuine spall, and does it stay
quiet on a healthy machine?

In [ ]:
def alarm_rate(spec, X):
    """Fraction of windows this spec would RAISE on, persistence included.

    Including the persistence rule is not fussiness.  A spec with a 9-of-5 rule
    parses, stores and scores perfectly well, and never raises anything: the
    machine is unprotected and the device reports no error at all.
    """
    mu = np.asarray(spec["scaler_mean"], dtype=float)
    sd = np.asarray(spec["scaler_scale"], dtype=float)
    if len(mu) != X.shape[1] or len(sd) != X.shape[1]:
        raise ValueError("feature count mismatch")
    if not np.all(np.isfinite(sd)) or np.any(np.abs(sd) < MIN_SD):
        raise FloatingPointError("degenerate scale")
    z = np.abs((X - mu) / sd).max(1)
    if not np.isfinite(z).all():
        raise FloatingPointError("non-finite score")
    over = (z > spec["threshold"]).astype(int)
    m = int(spec["persistence"]["m"])
    n = int(spec["persistence"]["n"])
    if m > n:
        return 0.0                       # unsatisfiable: the device is deaf
    k = np.convolve(over, np.ones(n, dtype=int), mode="full")[:len(over)]
    return float(np.mean(k >= m))


def outcome_of(store, kind, X_healthy, X_faulty, base_faulty,
               flood=0.5, deaf=0.10):
    """Classify what the machine is now living with.

    A structural check says whether an update is well formed.  This says
    whether the device can still do its job -- which is the only question the
    plant cares about.
    """
    if kind == "classifier":
        return "log only"                  # the classifier drives no actuator
    try:
        rh = alarm_rate(store.alarm, X_healthy)
        rf = alarm_rate(store.alarm, X_faulty)
    except Exception:
        return "crash"
    if rh > flood:
        return "alarm flood"
    if rf < deaf:
        return "deaf to the fault"
    if rf < 0.5 * base_faulty:
        return "degraded"
    return "no harm"

In [ ]:
BASE_FAULTY = alarm_rate(alarm0, FAULTY)
PATHS = ["one assignment", "structural gate", "+ shadow test"]
rows = []
for name, msg, expect in CASES:
    kind = msg.get("kind") if isinstance(msg, dict) else None
    row = {"case": name, "must": expect}
    s1 = SpecStore(copy.deepcopy(alarm0), copy.deepcopy(clf0), CUR_VER, CUR_VER)
    ok1, _ = apply_naive(s1, msg) if isinstance(msg, dict) else (False, "")
    s2 = SpecStore(copy.deepcopy(alarm0), copy.deepcopy(clf0), CUR_VER, CUR_VER)
    ok2, _ = s2.apply(msg, NOW)
    s3 = SpecStore(copy.deepcopy(alarm0), copy.deepcopy(clf0), CUR_VER, CUR_VER)
    ok3, why3 = (apply_verified(s3, msg, NOW, QUIET, ALARMED)
                 if isinstance(msg, dict) else (False, ""))
    for lbl, st, ok in (("one assignment", s1, ok1), ("structural gate", s2, ok2),
                        ("+ shadow test", s3, ok3)):
        row[lbl] = (outcome_of(st, kind, HEALTHY, FAULTY, BASE_FAULTY) if ok
                    else "refused")
    rows.append(row)

tbl = pd.DataFrame(rows).set_index("case")
print(tbl.to_string())
print()
for p in PATHS:
    right = sum(1 for r in rows if (r[p] != "refused") == (r["must"] == "accept"))
    print(f"{p:<18} correct on {right}/{len(rows)}")

### What that table says

**One assignment** — the version almost everyone writes first — gets
4 of 19 right and leaves the machine in a state it cannot
recover from in 9 cases: 3
crash on the first window, 3 go
**deaf to a genuine spall**, and
3 flood the plant with alarms on a
healthy machine. Note that the deaf ones are the dangerous ones and they are also the
quiet ones: nothing on the device, on the dashboard or in the log says anything is wrong.

**The structural gate** gets 17 of 19. Every check in it is a
few microseconds and each corresponds to something that has actually gone wrong in the
field. That is a very good return on about forty lines.

**The two it cannot catch** are the interesting ones:

In [ ]:
for r in rows:
    if r["structural gate"] != "refused" and r["+ shadow test"] == "refused":
        print(f"  {r['case']}")
        print(f"      structural gate: {r['structural gate']}")

## 5. What a structural check cannot see

Both of those updates have twelve finite means, twelve positive standard deviations, a
threshold in range, a sane persistence rule, a fresh timestamp and a correct checksum.
They are *well formed*. They are also wrong, in opposite directions:

* the spec refitted on data that **included the fault** learned that a spalled bearing
  is normal — it will never raise again;
* the spec refitted **during a heavy cut** learned that a loaded machine is the baseline
  — it raises on everything.

No amount of range checking finds either. The only thing that does is **running the
candidate before committing it**, on windows the device has kept — and it needs *two*
kinds of window, which is the part people miss.

In [ ]:
def spec_health(spec, X_quiet, X_alarmed=None, fa_budget=(0.0, 0.25),
                recall_min=0.30):
    """Run a candidate spec, in shadow, against windows the device kept.

    This is the check that catches an update which is perfectly well formed and
    still wrong.  It needs BOTH halves and the second one is the one people
    forget:

      X_quiet    windows from a stretch nothing happened in.  Catches a spec
                 that will flood -- fitted during a heavy cut, say.
      X_alarmed  the windows that made this device raise, last time it did.
                 Catches a spec that has gone DEAF, which is invisible on
                 healthy data by construction.

    The device has `X_alarmed` because Lecture 12's "on alarm + context" policy
    published the windows around every raise -- and, if you kept a copy on the
    device, this is what they were for.
    """
    try:
        rate = alarm_rate(spec, X_quiet)
    except Exception:
        return {"ok": False, "rate": None, "recall": None,
                "why": "unevaluable on the quiet stretch"}
    lo, hi = fa_budget
    if rate > hi:
        return {"ok": False, "rate": rate, "recall": None,
                "why": f"would raise on {rate:.0%} of a quiet stretch"}
    if rate < lo:
        return {"ok": False, "rate": rate, "recall": None, "why": "never raises"}

    rec = None
    if X_alarmed is not None and len(X_alarmed):
        try:
            rec = alarm_rate(spec, X_alarmed)
        except Exception:
            return {"ok": False, "rate": rate, "recall": None,
                    "why": "unevaluable on the retained alarm windows"}
        if rec < recall_min:
            return {"ok": False, "rate": rate, "recall": rec,
                    "why": f"deaf: raises on only {rec:.0%} of the windows that "
                           f"alarmed last time"}
    r_txt = "n/a" if rec is None else f"{rec:.0%}"
    return {"ok": True, "rate": rate, "recall": rec,
            "why": f"{rate:.1%} quiet, {r_txt} on retained alarms"}

In [ ]:
from downlink import spec_health
for nm, sp in (("the update we want", good_alarm),
               ("refit that absorbed the fault", overfit_alarm),
               ("refit during a heavy cut", loaded_alarm),
               ("the spec already running", alarm0)):
    h = spec_health(sp, QUIET, ALARMED)
    print(f"{nm:<32}{'PASS' if h['ok'] else 'FAIL':>6}   {h['why']}")

The quiet windows catch the spec that will flood. The **retained alarm windows** catch
the spec that has gone deaf — and deafness is invisible on healthy data by construction,
so without them the check is only half a check.

Where does a device get a set of windows that made it raise? It kept them. Lecture 12's
`on alarm + context` policy published a burst of raw feature vectors around every alarm,
and the reason to keep a copy in RAM was never stated at the time. This is it.

> A spec that has only been tested on quiet data has only been tested for one of the two
> ways it can be wrong.

## 6. Which artefact is dangerous to update

The classifier is 39 numbers. The alarm is 25. One of
them needs a shadow test, a rollback path and a named approver; the other can be swapped
whenever you like. Students reliably guess the wrong one.

In [ ]:
new_clf = json.load(open("lecture13_classifier_v4.json"))   # a legitimate retrain


def labels_of(c, Xm):
    Wc = np.array(c["coef"]) / np.array(c["scaler_scale"])
    b = (np.array(c["intercept"])
         - (np.array(c["coef"]) * (np.array(c["scaler_mean"])
                                   / np.array(c["scaler_scale"]))).sum(1))
    return np.array(c["classes"])[(Xm @ Wc.T + b).argmax(1)]


def raised_of(spec, Xm):
    m_, s_ = np.array(spec["scaler_mean"]), np.array(spec["scaler_scale"])
    over = (np.abs((Xm - m_) / s_).max(1) > spec["threshold"]).astype(int)
    return np.convolve(over, np.ones(5, int), "full")[:len(over)] >= 3


def score_of(spec, Xm):
    m_, s_ = np.array(spec["scaler_mean"]), np.array(spec["scaler_scale"])
    return np.abs((Xm - m_) / s_).max(1)


l_old, l_new = labels_of(clf0, X), labels_of(new_clf, X)
r_old, r_new = raised_of(alarm0, X), raised_of(good_alarm, X)
trans = lambda r: int(np.sum(r[1:] != r[:-1]))

print(f"{'':<26}{'labels changed':>16}{'actuator transitions':>22}{'windows raised':>16}")
print(f"{'swap the CLASSIFIER':<26}{np.mean(l_old != l_new):>15.1%}"
      f"{f'{trans(r_old)} -> {trans(r_old)}':>22}{f'{r_old.mean():.1%} -> {r_old.mean():.1%}':>16}")
print(f"{'swap the ALARM':<26}{0.0:>15.1%}"
      f"{f'{trans(r_old)} -> {trans(r_new)}':>22}{f'{r_old.mean():.1%} -> {r_new.mean():.1%}':>16}")

Swapping the classifier changed **3 % of the labels and
nothing else**. The classifier drives the log; a wrong label is a wrong word in a file.

Swapping the alarm changed **no labels at all** and rewrote the entire actuator history:
371 transitions became 241, and the
fraction of windows under alarm went from 27 % to
52 %.

> **Gate an update in proportion to what it can break, not to how big it is.**

There is a second consequence, and it bites six months later.

In [ ]:
s_old, s_new = score_of(alarm0, X), score_of(good_alarm, X)
i = int(np.searchsorted(t, 15 * 60.0))
before, after = np.median(s_old[i - 160:i]), np.median(s_new[i:i + 160])
print(f"reported anomaly score just before the commit: {before:.2f}")
print(f"reported anomaly score just after  the commit: {after:.2f}")
print(f"a factor of {after / before:.1f}, and the machine did not change at all")
print()
print(f"the two thresholds: {alarm0['threshold']:.2f} and {good_alarm['threshold']:.2f} "
      "-- almost identical")

The score is a z-score *against a baseline*, so when the baseline changes the score
changes meaning. The threshold barely moved; the number being compared to it moved by a
factor of three.

Anything that accumulates the score across a spec change is now telling a story that did
not happen: Lecture 12's `a_max` in the heartbeat, the fleet trend of Lecture 12 §7, and
every dashboard chart of the last two years.

**Carry the spec version in the uplink**, and re-baseline the history at every version
boundary. It costs about six bytes and it is the difference between a trend you can read
and a trend with an unexplained step in it.

## 7. When the sensor lies

Everything so far has assumed the twelve features describe the machine. They describe
the *signal*, and the signal comes from an accelerometer with a bolt, a cable, a bias, a
full-scale setting and an adhesive bond — every one of which fails eventually.

Four ordinary faults, none of them exotic.

In [ ]:
ADC_FULL_SCALE = 2.0          # g, the range the sensor is configured for
LSB = ADC_FULL_SCALE / 32768.0


# ------------------------------------------------------------- the faults
def fault_stuck(x, rng, at=0.0):
    """The bus read stops updating.  A dead channel is not a quiet channel: it
    is the last value, forever, plus the noise of the converter itself."""
    y = x.copy()
    i = int(at * len(x))
    y[i:] = x[i] + rng.normal(0, LSB, len(x) - i)
    return y


def fault_clip(x, rng, level=0.55):
    """The range is set too low.  The signal is intact in the middle and gone
    at the edges -- which is where the bearing impulses live."""
    return np.clip(x, -level, level)


def fault_loose(x, rng, amp=1.40, f_mount=850.0, rate=90.0, fs=2000.0):
    """The sensor rattles on its own mounting bolt.

    This is the nastiest of the four because it looks like exactly the thing we
    trained the model to find: a train of sharp, decaying impulses.
    """
    n = len(x)
    t = np.arange(n) / fs
    y = x.copy()
    period = fs / rate
    k = 0
    while True:
        i = int(k * period * (1.0 + rng.normal(0, 0.05)))
        if i >= n:
            break
        tail = np.arange(n - i) / fs
        y[i:] += (amp * rng.uniform(0.6, 1.4) * np.exp(-tail / 0.0014)
                  * np.sin(2 * np.pi * f_mount * tail))
        k += 1
    return y


def fault_gain(x, rng, g=0.25):
    """The bonding has aged, or somebody wrote 16 g where 2 g was meant.
    Everything scales.  The normalised band energies do not move at all."""
    return g * x


FAULTS = {
    "stuck channel": fault_stuck,
    "clipped range": fault_clip,
    "loose sensor mount": fault_loose,
    "gain drift ×0.25": fault_gain,
}

In [ ]:
from cloud import EdgeDevice
import sensors

dev = EdgeDevice("lecture9_model.json", "lecture10_alarm.json")
rng = np.random.default_rng(77)
p_ok = rig._run_params(rng, "normal")
x_ok = rig._synth(rng, p_ok, 8.0, rig.FS)
p_bad = rig._run_params(rng, "bearing")
p_bad["a_imp"] = 0.9
x_bad = rig._synth(rng, p_bad, 8.0, rig.FS)


def adc(x, full=2.0):
    '''What the device's converter actually hands to the code.'''
    lsb = full / 32768.0
    return np.round(np.clip(x, -full, full) / lsb) * lsb


def verdict(x, tag, truth):
    Wm = rig.frame(adc(x))
    F = rig.features_of(Wm)
    D = dev.run(F)
    return {"truth": truth, "sensor": tag,
            "label": max(set(D["label"]), key=list(D["label"]).count),
            "margin": round(float(np.median(D["margin"])), 2),
            "score": round(float(np.median(D["score"])), 1),
            "raised": round(float(D["raised"].mean()), 3),
            "rms": round(float(np.median(F[:, 1])), 3),
            "kurt": round(float(np.median(F[:, 5])), 2),
            "crest": round(float(np.median(F[:, 4])), 2)}


rowsS = [verdict(x_ok, "good sensor", "healthy"),
         verdict(x_bad, "good sensor", "SPALL")]
for nm, fn in sensors.FAULTS.items():
    rowsS.append(verdict(fn(x_ok, rng), nm, "healthy"))
rowsS.append(verdict(sensors.fault_clip(x_bad, rng), "clipped range", "SPALL"))
rowsS.append(verdict(sensors.fault_gain(x_bad, rng), "gain drift ×0.25", "SPALL"))
print(pd.DataFrame(rowsS).to_string(index=False))

Read the rows in pairs.

**The stuck channel** produces the most confident classification in the whole course: the
label is `normal` with a margin of
9.8, against
1.4 on a genuinely healthy machine — the Lecture 11 lesson
about margins on unfamiliar inputs, arriving again from a different direction.
Meanwhile the alarm screams
2240. The two
models disagree violently and neither is right; the sensor is simply not connected.

**The loose mounting bolt** is the nasty one. A sensor rattling on its own bolt produces
a train of sharp decaying impulses — which is exactly what we trained the model to find.
It calls a perfectly healthy machine `bearing` and raises on
29 %
of windows. That is a work order, a strip-down and nothing wrong with the bearing.

**And the row to remember:**

In [ ]:
h = [r for r in rowsS if r["truth"] == "healthy" and r["sensor"] == "good sensor"][0]
s = [r for r in rowsS if r["truth"] == "SPALL" and r["sensor"] == "clipped range"][0]
print("a healthy machine, good sensor   kurt %.2f  crest %.2f  -> %s" %
      (h["kurt"], h["crest"], h["label"]))
print("a SPALLED bearing, clipped range kurt %.2f  crest %.2f  -> %s" %
      (s["kurt"], s["crest"], s["label"]))
print()
print("the broken machine looks CALMER than the healthy one, and the alarm raised on "
      f"{s['raised']:.0%} of windows")

Clipping removes the peaks, and the peaks are the fault. Kurtosis and crest factor are
both peak statistics, so a sensor that cannot see peaks does not make the fault noisy —
it **erases** it, and leaves a signal that scores better than health.

This is the difference between a sensor fault and noise, and it is why "the model will
degrade gracefully" is not true here. Noise makes a model uncertain. A sensor fault makes
it confident and wrong.

## 8. The validity gate

The fix is not a better model. It is five comparisons on the raw window, before any
feature is computed, asking one question: **is this plausibly a working accelerometer on
this machine?**

Every limit is a property of the *installation*, not of the datasheet, and has to come
from that installation's own history — the same discipline as the Lecture 10 baseline.
Copying these numbers onto a different rig is the mistake the file exists to prevent.

In [ ]:
# --------------------------------------------------------------- the gate
# Every limit here is a property of the INSTALLATION, not of the machine, and
# every one is checkable in one pass over the raw window.
SAT_FRAC = 0.010              # more than 1 % of samples pinned at one value
FLAT_TOL = 3.0                # "pinned" means within 3 LSB of the extreme --
                              # a natural signal never repeats its peak, a
                              # clipped one repeats it dozens of times
DEAD_PTP = 0.02               # g: a running machine is never this quiet
MAX_DC = 0.50                 # g: the accelerometer's bias is specified
RMS_RANGE = (0.12, 4.0)       # g: from THIS installation's own history,
                              # not from the datasheet -- same discipline as
                              # the alarm baseline
MIN_E1X = 0.02                # a running rotor always has a shaft line


def validity(buf, fs=2000.0, full_scale=ADC_FULL_SCALE, e_1x=None):
    """Is this window plausibly from a working accelerometer on this machine?

    Returns (ok, reason).  Four of the five checks are two comparisons per
    sample; the fifth reuses a feature the device computes anyway.

    The order matters: report the cheapest, most specific failure first, because
    the reason is what a technician acts on.
    """
    x = np.asarray(buf, dtype=float)
    hi, lo = float(x.max()), float(x.min())
    span = hi - lo
    if span < DEAD_PTP:
        return False, "dead channel"
    # Clipping is not detected against the CONFIGURED range -- a signal can be
    # clipped by an amplifier, a cable or a badly chosen full scale, and all of
    # them look the same: a pile of samples sitting on one value.  A natural
    # signal never repeats its own peak; a clipped one repeats it dozens of
    # times.
    tol = FLAT_TOL * LSB
    pinned = max(np.mean(x >= hi - tol), np.mean(x <= lo + tol))
    if pinned > SAT_FRAC:
        return False, "clipped"
    mean = float(x.mean())
    if abs(mean) > MAX_DC:
        return False, "dc offset"
    rms = float(np.sqrt(np.mean((x - mean) ** 2)))
    if not (RMS_RANGE[0] <= rms <= RMS_RANGE[1]):
        return False, "level implausible"
    if e_1x is not None and e_1x < MIN_E1X:
        return False, "no shaft line"
    return True, "ok"

In [ ]:
I1X = rig.FEATURE_NAMES.index("e_1x")


def gated(x, tag, truth):
    Wm = rig.frame(adc(x))
    F = rig.features_of(Wm)
    ok, why = sensors.validity_table(Wm, e1x=F[:, I1X])
    from collections import Counter
    top = Counter(why[~ok]).most_common(1)
    return {"truth": truth, "sensor": tag,
            "rejected": round(float(1 - ok.mean()), 3),
            "reason": top[0][0] if top else "-"}


g = [gated(x_ok, "good sensor", "healthy"), gated(x_bad, "good sensor", "SPALL")]
for nm, fn in sensors.FAULTS.items():
    g.append(gated(fn(x_ok, rng), nm, "healthy"))
g.append(gated(sensors.fault_clip(x_bad, rng), "clipped range", "SPALL"))
g.append(gated(sensors.fault_gain(x_bad, rng), "gain drift ×0.25", "SPALL"))
print(pd.DataFrame(g).to_string(index=False))

In [ ]:
# what it costs: 30 runs of genuinely good data, all three machine states
fr = []
for st_ in ("normal", "imbalance", "bearing"):
    for k in range(10):
        rr = np.random.default_rng(400 + 37 * k + len(st_))
        q = rig._run_params(rr, st_)
        Wg = rig.frame(adc(rig._synth(rr, q, 4.0, rig.FS)))
        Fg = rig.features_of(Wg)
        okg, _ = sensors.validity_table(Wg, e1x=Fg[:, I1X])
        fr.append(1 - okg.mean())
print(f"false rejections over {len(fr)} good runs: mean {np.mean(fr):.4f}, "
      f"max {np.max(fr):.4f}")

Five comparisons catch 5 of the 6 sensor-fault cases
at a measured false-rejection rate of **0.0000** over
30 runs of healthy, imbalance and bearing machines. Under MicroPython it
costs **106 µs**, about 8 % of the pipeline — the
cheapest insurance in the course.

And it misses one: **loose sensor mount**.

That is not a bug to be fixed by adding a sixth comparison. A sensor rattling on its own
bolt produces a signal that is, by every cheap measure, a genuine impulsive vibration —
because it is one. Telling it apart from a bearing defect needs the *repetition rate*:
a spall repeats at a bearing order of shaft speed (here 3.6×, about 108 Hz), a loose bolt
repeats at whatever the mount resonance wants. That needs a tachometer or a cepstrum, and
it is beyond what this device can do.

So the honest architecture is:

1. the gate rejects what is cheaply impossible;
2. the cloud's fleet comparison flags what is merely unlikely (Lecture 12 §7 — one
   machine suddenly impulsive while eleven peers are not);
3. and a **human with a spanner** closes the remaining gap.

The third step is not a failure of the design. It is the design.

## 9. Degraded modes, and the log

| what fails | what the device does | protection |
|---|---|---|
| the network drops | keep deciding, queue the report | none lost |
| the broker refuses the connection | keep deciding, back off, retry | none lost |
| an update fails validation | keep the running spec, ack the reason | none lost |
| a committed spec misbehaves | roll back to `last_good`, tell the cloud | none lost |
| the classifier is uncertain | label `"uncertain"`; the alarm is unaffected | none lost |
| the sensor fails the gate | hold the actuator, publish `sensor-fault` | **degraded** |
| feature extraction raises | hold the actuator, log the window | **degraded** |
| the spec is past its expiry date | keep using it, flag it, escalate | **degraded** |

Not one row says *stop deciding*. A device that stops protecting the machine when the
network does is not an edge device; it is a sensor with extra steps.

Three of the rows are marked degraded, and the distinction matters when somebody asks
what happened: in those three the device is still *safe* — it holds its last actuator
state — but it is no longer *protecting*, and somebody has to know that. "Hold and be
quiet about it" is the one behaviour that is never acceptable.

### What the log must contain

You will be asked, months later, why the line stopped. The log has to answer it without
anybody guessing:

| | why |
|---|---|
| window timestamp | to line the event up with production records |
| the twelve features | so the decision can be recomputed exactly |
| spec versions, both | so you know *which* model decided |
| the verdict, margin, score and driver | what it decided, and on what evidence |
| gate result | whether the input was even trusted |
| actuator state before and after | what the machine actually did |
| for an update: version, decision, reason, approver | who changed what, and when |

That is about 200 bytes a decision and you do not keep all of them — keep every event,
every update, and a sample of the rest. **A decision you cannot reconstruct is a
decision you cannot defend.**

## 10. The device code

The shipping version is in the lab package; these are the two pieces that matter. Read
`service()` for the order — validate, verify, commit, ack — and note that the MQTT
callback does nothing except set a flag. Parsing JSON inside an interrupt-adjacent
callback, between two windows of a control loop, is how a device misses its deadline.

```python
def on_model_message(topic, payload):
    """MQTT callback.  Does the minimum and returns: no parsing in here."""
    global _pending
    _pending = payload


def service(now, publish=None):
    """Call from the main loop, between windows.  Returns (accepted, reason)."""
    global _pending
    if _pending is None:
        return None
    raw, _pending = _pending, None
    try:
        msg = json.loads(raw)
    except Exception:
        return _ack(publish, None, None, False, "unparseable")

    p, why = validate(msg, now)
    if p is None:
        return _ack(publish, msg.get("kind"), msg.get("ver"), False, why)

    if msg["kind"] == "alarm":
        ok, why = verify(p)
        if not ok:
            return _ack(publish, "alarm", msg["ver"], False, "shadow: " + why)
        _last_good["alarm"] = (alarm.MEAN, alarm.INV_SD, alarm.THRESHOLD)
        inv = [1.0 / v for v in p["scaler_scale"]]
        alarm.MEAN = tuple(p["scaler_mean"])          # commit: three names
        alarm.INV_SD = tuple(inv)
        alarm.THRESHOLD = p["threshold"]
        alarm.reset()                 # the ring holds scores from the OLD spec
    else:
        _last_good["classifier"] = (model.COEF, model.INTERCEPT)
        model.COEF = tuple(tuple(r) for r in p["coef"])
        model.INTERCEPT = tuple(p["intercept"])

    _ver[msg["kind"]] = msg["ver"]
    gc.collect()
    return _ack(publish, msg["kind"], msg["ver"], True, "committed: " + why)
```

And the gate, in the form that runs on the device — one pass, no allocation, no imports:

```python
def validity(buf, e_1x=None):
    """(ok, reason).  Never allocates, never raises."""
    n = len(buf)
    lo = hi = buf[0]
    total = 0.0
    for v in buf:
        total += v
        if v < lo:
            lo = v
        if v > hi:
            hi = v
    span = hi - lo
    if span < DEAD_PTP:
        return False, "dead channel"

    n_hi = 0
    n_lo = 0
    s2 = 0.0
    mean = total / n
    for v in buf:
        if v >= hi - FLAT_TOL:
            n_hi += 1
        if v <= lo + FLAT_TOL:
            n_lo += 1
        d = v - mean
        s2 += d * d
    pinned = (n_hi if n_hi > n_lo else n_lo) / n
    if pinned > SAT_FRAC:
        return False, "clipped"

    if mean > MAX_DC or mean < -MAX_DC:
        return False, "dc offset"
    rms = (s2 / n) ** 0.5
    if rms < RMS_LO or rms > RMS_HI:
        return False, "level implausible"
    if e_1x is not None and e_1x < MIN_E1X:
        return False, "no shaft line"
    return True, "ok"
```

## The protocol

Closing the loop, in the order the steps must happen:

1. **Work out the latency budget first.** If the device path already uses most of it,
   the cloud is not going in the loop and everything else follows from that.
2. **Version every artefact**, and refuse anything not strictly newer.
3. **Put a shelf life on the spec** and enforce it on the device, not in a wiki.
4. **Validate structure, integrity and ranges** before the candidate touches anything.
5. **Stage into a standby slot.** Never modify what is running.
6. **Verify by running it** — on quiet windows *and* on windows that made this device
   raise. One without the other is half a check.
7. **Commit with a single assignment**, and reset anything that accumulated under the
   old spec.
8. **Acknowledge, with the reason.** A silent rejection is indistinguishable from a lost
   message.
9. **Keep `last_good`** and the ability to roll back without a human present.
10. **Gate the sensor before the model.** A good model on a bad signal is confidently
    wrong, and that is worse than being uncertain.
11. **Never stop deciding.** Degrade, hold, log, escalate — but keep the loop running.
12. **Log enough to reconstruct the decision**, including which spec version made it and
    who approved that spec.

## Exercises

1. **Add a case to the gauntlet.** Write an update that this notebook's gates accept and
   that you believe would still be harmful. If you succeed, add the check that catches it
   and say what that check costs. If you cannot, say what stopped you — that is a
   result too.

2. **The verify buffers.** Section 5 uses 1200 quiet windows and
   400 retained alarm windows. Sweep both sizes from 8 to 400 and find
   the smallest pair that still refuses both well-formed-but-wrong specs. How much RAM
   is that on the device, in the `array('f')` terms of Lecture 11?

3. **Rollback, properly.** `SpecStore.rollback` exists but nothing calls it. Write the
   rule that *would* call it: the device has committed a spec, and some number of windows
   later decides it was a mistake. What is the statistic, what is the threshold, and how
   do you avoid rolling back a spec that is correctly reporting a machine that really has
   started to fail?

4. **The score discontinuity.** Implement the fix from section 6: carry `spec_ver` in the
   uplink summary, and write the cloud-side code that re-baselines a trend at each
   version boundary. Show it on the Lecture 12 fleet trend with a spec change inserted at
   month 12.

5. **A fifth sensor fault.** Add intermittent connection — the signal is correct for a
   few windows, then a few windows of nothing, repeatedly. Which gate check catches it,
   and at what duty cycle does it start slipping through?

6. **Gate limits from history.** The limits in `validity.py` were chosen by hand. Derive
   them instead from a month of this machine's own healthy windows, the way Lecture 10
   derived its threshold, and state the false-rejection rate you are buying.

7. **The loose bolt, caught.** Section 8 says the repetition rate distinguishes a loose
   mount from a spall. Implement it: estimate the impulse repetition rate from the
   envelope autocorrelation and test whether it is near a bearing order of the shaft
   rate. Measure the cost, and decide honestly whether it belongs on the device.

8. **Towards Lecture 14.** Write the sequence of events for the integrated run: a healthy
   machine, a spec update pushed and accepted, a fault injected, the alarm raising, the
   actuator firing, the cloud alerting, and the WiFi pulled out halfway through. At each
   step, say what you would have to see to believe the system worked.

## References

**Updating devices in the field**

* Asokan, N. et al. (2018). *ASSURED: Architecture for secure software update of realistic
  embedded devices.* IEEE Transactions on CAD, 37(11). The staged-commit pattern of
  section 3, with the threat model this lecture deliberately leaves out.
* IETF RFC 9019 (2021). *A Firmware Update Architecture for Internet of Things.*
  Manifests, versioning and why the checksum is not a signature.
* Beyer, B., Jones, C., Petoff, J. and Murphy, N. R. (2016). *Site Reliability
  Engineering*, chapters on release engineering and canarying. The cloud version of the
  same argument.

**Sensor validation**

* Dunia, R., Qin, S. J., Edgar, T. F. and McAvoy, T. J. (1996). *Identification of faulty
  sensors using principal component analysis.* AIChE Journal, 42(10), 2797–2812.
* Isermann, R. (2006). *Fault-Diagnosis Systems: An Introduction from Fault Detection to
  Fault Tolerance.* Springer. Chapter 1 on the distinction this lecture's section 7
  turns on: a fault in the process against a fault in the measurement.
* Randall, R. B. and Antoni, J. (2011). *Rolling element bearing diagnostics — a
  tutorial.* Mechanical Systems and Signal Processing, 25(2), 485–520. The envelope and
  repetition-rate analysis that exercise 7 asks for.

**Safety, alarms and accountability**

* Leveson, N. G. (2011). *Engineering a Safer World: Systems Thinking Applied to Safety.*
  MIT Press. Why "the component worked as specified" is not the same as "the system was
  safe".
* ISA-18.2 (2016). *Management of Alarm Systems for the Process Industries.*
* Amodei, D., Olah, C., Steinhardt, J., Christiano, P., Schulman, J. and Mané, D. (2016).
  *Concrete problems in AI safety.* arXiv:1606.06565. Distributional shift and safe
  interruptibility, in the language this course has been using all term.

**Tools**

* MicroPython `hashlib` and `ubinascii`:
  https://docs.micropython.org/en/latest/library/hashlib.html
* NETPIE private topics: https://docs.netpie.io/